In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 39. Week 27 — Kalman filtering, smoothing, missing data, and DNS

## 学習目標

- linear-Gaussian predict/update recursionを導出できる
- filteringとsmoothingの情報集合を区別できる
- missing tenorでupdate rowを落とす処理を検証できる
- fixed-decay Dynamic Nelson–Siegelをfitし5公表日先を予測できる

## 前提知識

- multivariate Gaussian conditioning
- Week 26のNelson–Siegel factorsとVAR

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 39


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Linear-Gaussian state space

$$
x_t=c+Fx_{t-1}+\eta_t,\qquad y_t=Hx_t+\varepsilon_t,
$$

$$
K_t=P_{t|t-1}H^\top(HP_{t|t-1}H^\top+R)^{-1}.
$$

実装はinverseを作らずlinear solveを使う。filter (p(x_t\mid y_{1:t})) はforecast originで利用可能、smoother (p(x_t\mid y_{1:T})) はretrospective diagnostic専用である。

In [4]:
training_change = np.diff(curve_yields[train_mask, 3])
level_q = np.var(training_change, ddof=1)
level_r = max(0.25 * level_q, 1e-8)
available = curve_dates <= validation_end_date
local_level = qt.kalman_filter(
    curve_yields[available, 3],
    [[1.0]],
    [[1.0]],
    [[level_q]],
    [[level_r]],
    [curve_yields[0, 3]],
    [[1.0]],
)
local_smoother = qt.kalman_smoother(local_level, [[1.0]])
assert np.allclose(local_smoother.smoothed_means[-1], local_level.filtered_means[-1])

fig = go.Figure()
fig.add_scatter(x=curve_dates[available], y=curve_yields[available, 3], name="observed 10y", mode="lines")
fig.add_scatter(x=curve_dates[available], y=local_level.filtered_means[:, 0], name="filtered", mode="lines")
fig.add_scatter(x=curve_dates[available], y=local_smoother.smoothed_means[:, 0], name="smoothed", mode="lines")
fig.update_layout(title="Filtering is online; smoothing is retrospective", yaxis_title="10y CMT (%)", template="plotly_white")
fig.show()

## 2. Missing-observation audit

validationの10年tenorを規則的にblankへ置換する。これは市場値の擬似生成ではなく、欠測処理を既知マスクで検証するstress testである。

In [5]:
missing_panel = curve_yields[available].copy()
validation_rows = np.flatnonzero((curve_dates[available] > train_end_date) & (curve_dates[available] <= validation_end_date))
missing_rows = validation_rows[::17]
missing_panel[missing_rows, 3] = np.nan

dns_model = qt.fit_dynamic_nelson_siegel(curve_yields[train_mask], maturity_years, decay=0.5)
complete_filter = qt.filter_dynamic_nelson_siegel(dns_model, curve_yields[available])
missing_filter = qt.filter_dynamic_nelson_siegel(dns_model, missing_panel)
assert not np.any(missing_filter.observed_mask[missing_rows, 3])
missing_audit = pd.DataFrame(
    {
        "metric": ["blanked 10y rows", "complete log likelihood", "missing-panel log likelihood", "mean filtered-state difference"],
        "value": [
            len(missing_rows),
            complete_filter.log_likelihood,
            missing_filter.log_likelihood,
            np.mean(np.linalg.norm(complete_filter.filtered_means - missing_filter.filtered_means, axis=1)),
        ],
    }
)
display(missing_audit)

,metric,value
0,blanked 10y rows,33.000000
1,complete log likelihood,4101.019415
2,missing-panel log likelihood,4159.582429
3,mean filtered-state difference,0.000800


## 3. Five-publication DNS forecast

In [6]:
last_validation_origin = np.flatnonzero(curve_dates <= validation_end_date)[-6]
origin_filter = qt.filter_dynamic_nelson_siegel(dns_model, curve_yields[: last_validation_origin + 1])
predictive = qt.forecast_dynamic_nelson_siegel(
    dns_model,
    origin_filter.filtered_means[-1],
    origin_filter.filtered_covariances[-1],
    5,
)
display(
    pd.DataFrame(
        {
            "tenor": qt.DEFAULT_TENORS,
            "forecast_percent": predictive.mean,
            "actual_percent": curve_yields[last_validation_origin + 5],
            "forecast_standard_deviation_bp": 100.0 * np.sqrt(np.diag(predictive.covariance)),
        }
    )
)

,tenor,forecast_percent,actual_percent,forecast_standard_deviation_bp
0,3m,5.590450,5.60,5.347277
1,2y,4.939937,5.14,8.939796
2,5y,4.639595,4.95,10.063969
3,10y,4.624384,4.98,10.733224
4,30y,4.698832,5.11,11.084372


## 4. 失敗モード

- smoothed stateをhistorical forecast originへ戻して使う
- missing値をzero yieldとしてupdateする
- (Q,R,F) をouter testで調整する
- covarianceのPSD、innovation、log likelihoodを監査しない
- fixed decayのtwo-step DNSをjoint maximum likelihoodと呼ぶ

## 5. 段階別演習

### 基礎

1. scalar local-level filterのgainを導出せよ。
2. 全tenor missingの日にpredictだけが行われることを確認せよ。

### 標準

3. missing率を1%、10%、30%へ変えfilter感応度を測れ。
4. decay 0.25/0.5/1.0をvalidation likelihoodで比較するprotocolを書け。

### 研究

5. EMで (Q,R) を推定するときのinitializationとlocal optimum監査を設計せよ。

## 6. Exit Criteria

- [ ] filterとsmootherの条件付け集合を書ける
- [ ] missing rowを観測方程式から除外した
- [ ] covarianceを対称PSDとして監査した
- [ ] forecast originではfiltered stateだけを使った
- [ ] DNS two-step estimationの限界を明記した

## 7. 出典


- [Kalman (1960), A New Approach to Linear Filtering and Prediction Problems](https://people.math.harvard.edu/archive/116_fall_03/handouts/Kalman1960.pdf)
- [Särkkä and Svensson, Bayesian Filtering and Smoothing, 2nd ed.](https://users.aalto.fi/~ssarkka/pub/bfs_book_2023_online.pdf)
- [Diebold and Li, Forecasting the Term Structure of Government Bond Yields](https://www.nber.org/papers/w10048.pdf)